# Analyze Numbered Sync-Check Timing Sweep Results

This notebook scans `runs/camera/**/summary.json`, `metadata.json`, `timing.json`, and `accumulated.npy` from `run_camera_sync_check.sh`. It ranks timing settings and also builds per-trigger rows with the `expected_number`, so you can inspect whether frame `i` looks like the number that should have been visible for trigger `i`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUN_ROOT = Path("runs/camera")
RUN_ROOT.resolve()

In [ ]:
def load_json(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def mean_or_nan(values):
    if values is None or len(values) == 0:
        return np.nan
    return float(np.mean(values))


def sum_or_nan(values):
    if values is None or len(values) == 0:
        return np.nan
    return float(np.sum(values))

In [ ]:
run_rows = []
frame_rows = []
for summary_path in sorted(RUN_ROOT.rglob("summary.json")):
    run_dir = summary_path.parent
    summary = load_json(summary_path)
    metadata = load_json(run_dir / "metadata.json")
    if metadata.get("mode") != "sync-check":
        continue
    timing = load_json(run_dir / "timing.json")
    number_sequence = metadata.get("number_sequence", [])
    trigger_policy = metadata.get("trigger_policy", timing)

    in_counts = summary.get("events_per_accumulation_window", [])
    pre_counts = summary.get("events_per_pre_trigger_window", [])
    post_counts = summary.get("events_per_post_window", [])
    abs_sums = summary.get("accumulated_abs_sums", [])
    nonzero = summary.get("accumulated_nonzero_pixels", [])
    frame_count = max(len(in_counts), len(abs_sums), len(nonzero))

    for i in range(frame_count):
        expected_number = number_sequence[i % len(number_sequence)] if number_sequence else None
        in_count = in_counts[i] if i < len(in_counts) else np.nan
        pre_count = pre_counts[i] if i < len(pre_counts) else np.nan
        post_count = post_counts[i] if i < len(post_counts) else np.nan
        abs_sum = abs_sums[i] if i < len(abs_sums) else np.nan
        crossover_score = (float(pre_count or 0) + float(post_count or 0)) / (float(in_count or 0) + 1.0)
        frame_rows.append({
            "run_dir": str(run_dir),
            "run_name": run_dir.name,
            "frame_index": i,
            "expected_number": expected_number,
            "in_window_events": in_count,
            "pre_window_events": pre_count,
            "post_window_events": post_count,
            "accumulated_abs_sum": abs_sum,
            "crossover_score": crossover_score,
        })

    run_rows.append({
        "run_dir": str(run_dir),
        "run_name": run_dir.name,
        "number_sequence": ",".join(str(n) for n in number_sequence),
        "number_size_px": metadata.get("number_size_px"),
        "b_dot_radius": metadata.get("b_dot_radius"),
        "exposure_us": metadata.get("exposure_us"),
        "dark_time_us": metadata.get("dark_time_us"),
        "trigger_rising_delay_us": trigger_policy["rising_delay_us"],
        "window_us": summary.get("window_us"),
        "actual_trigger_count": summary.get("actual_trigger_count"),
        "expected_trigger_count": metadata.get("expected_trigger_count"),
        "event_count": summary.get("event_count"),
        "in_window_mean": mean_or_nan(in_counts),
        "pre_window_mean": mean_or_nan(pre_counts),
        "post_window_mean": mean_or_nan(post_counts),
        "in_window_sum": sum_or_nan(in_counts),
        "pre_window_sum": sum_or_nan(pre_counts),
        "post_window_sum": sum_or_nan(post_counts),
        "accumulated_abs_sum_mean": mean_or_nan(abs_sums),
        "accumulated_nonzero_mean": mean_or_nan(nonzero),
        "summary_path": str(summary_path),
    })

df = pd.DataFrame(run_rows)
frames = pd.DataFrame(frame_rows)
print(f"Loaded {len(df)} sync-check runs and {len(frames)} frame rows from {RUN_ROOT}")
df.head()

In [ ]:
if df.empty:
    raise RuntimeError("No sync-check summaries found. Run the commands from 01_generate_timing_sweep_commands.ipynb first.")

eps = 1.0
df["pre_ratio"] = df["pre_window_sum"].fillna(0) / (df["in_window_sum"].fillna(0) + eps)
df["post_ratio"] = df["post_window_sum"].fillna(0) / (df["in_window_sum"].fillna(0) + eps)
df["crossover_score"] = df["pre_ratio"] + df["post_ratio"]
df["window_signal"] = np.log1p(df["in_window_sum"].fillna(0))
df["image_signal"] = np.log1p(df["accumulated_abs_sum_mean"].fillna(0))
df["trigger_completeness"] = df["actual_trigger_count"].fillna(0) / df["expected_trigger_count"].replace(0, np.nan)

# timing_score rewards complete in-window signal and penalizes nearby leakage/crossover.
df["timing_score"] = (
    df["window_signal"]
    + 0.35 * df["image_signal"]
    + 1.0 * df["trigger_completeness"].fillna(0)
    - 1.50 * df["pre_ratio"].fillna(0)
    - 1.00 * df["post_ratio"].fillna(0)
)

ranked_runs = df.sort_values("timing_score", ascending=False).reset_index(drop=True)
ranked_runs[[
    "run_name", "timing_score", "crossover_score", "exposure_us", "dark_time_us",
    "trigger_rising_delay_us", "number_sequence", "number_size_px", "b_dot_radius",
    "actual_trigger_count", "expected_trigger_count", "in_window_mean", "pre_window_mean", "post_window_mean",
]].head(20)

## Per-Number / Per-Frame Inspection

The frame table maps each trigger index to the `expected_number`. Use this to spot number crossover: high pre/post counts around a frame suggest the camera accumulation window is catching events from a neighboring numbered bitplane.

In [ ]:
if not frames.empty:
    display(frames.sort_values(["run_name", "frame_index"]).head(30))
    display(
        frames.groupby("expected_number", dropna=False)[[
            "in_window_events", "pre_window_events", "post_window_events", "crossover_score"
        ]].mean()
    )

In [ ]:
for exposure, group in ranked_runs.groupby("exposure_us", dropna=False):
    pivot = group.pivot_table(
        index="dark_time_us",
        columns="trigger_rising_delay_us",
        values="timing_score",
        aggfunc="mean",
    )
    if pivot.empty:
        continue
    fig, ax = plt.subplots(figsize=(8, 4))
    image = ax.imshow(pivot.values, aspect="auto", origin="lower")
    ax.set_title(f"sync-check timing_score, exposure_us={exposure}")
    ax.set_xlabel("trigger_rising_delay_us")
    ax.set_ylabel("dark_time_us")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    fig.colorbar(image, ax=ax)
    plt.show()

In [ ]:
columns = [
    "run_name", "timing_score", "crossover_score", "exposure_us", "dark_time_us",
    "trigger_rising_delay_us", "number_sequence", "number_size_px", "b_dot_radius",
    "window_us", "actual_trigger_count", "expected_trigger_count", "run_dir",
]
Path("runs").mkdir(exist_ok=True)
ranked_runs[columns].to_csv("runs/sync_timing_sweep_ranked_runs.csv", index=False)
frames.to_csv("runs/sync_timing_sweep_frame_rows.csv", index=False)
print("Wrote runs/sync_timing_sweep_ranked_runs.csv")
print("Wrote runs/sync_timing_sweep_frame_rows.csv")